In [ ]:
import torch
import matplotlib.pyplot as plt

from bhtrace.geometry import  Photon, Observer
import bhtrace.geometry.spacetime as st
from bhtrace.tracing import PTracer
from bhtrace.scenarios import Lensing
from bhtrace.graphics.lensing import lensing_curve
import bhtrace.utils.units as bhU
import bhtrace.physics.electrodynamics as bhE

In [ ]:
N = 128 # Number of initial rays
B_SPAN = 20
B0 = 0.5
OBS_R = 20.0
OBS_INCLINATION = torch.pi / 2
KERR_A = 0.6
TRACE_T = 50
TRACE_NSTEPS = 256
TRACE_EPS = 1e-5
ODE_METHOD = 'VCABM4'
N_SPLITS = 2

system = bhU.NaturalGeometric(M=2.2)

base = st.KerrBL(a=0.6)
model = bhE.EulerHeisenberg(system, scale=100.0)
field = bhE.fields.SplitMonopole(B0=bhU.schwinger_B.to(system).f)
eff = st.EffectiveGeometry(base, model, field)

SPACETIMES = {
    'base': base,
    'eh_eff': st.EffectiveGeometry(base, model, field),
}

In [ ]:
outp = {}

for name, spacetime in SPACETIMES.items():

    particle = Photon(spacetime)
    tracer = PTracer(eps=TRACE_EPS, ode_method=ODE_METHOD)
    tracer.to(dtype=torch.float32, dev='cuda')
    tracer.__const_dx__ = True

    observer = Observer(spacetime, r=OBS_R, inclination=OBS_INCLINATION)
    observer.generate_net('line', net_rng=(N,), net_size=(B_SPAN, 0))

    lensing = Lensing(tracer, observer, particle)

    x, dphi, traj = lensing.forward(nsplits=N_SPLITS, T=TRACE_T, nsteps=TRACE_NSTEPS)

    outp[name] = x, dphi, traj

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(16, 8))
fig.suptitle('Lensing comparison for Kerr BH and Kerr BH in effective geometry, sourced by Euler-Heisenberg NED')

for key, value in outp.items():
    lensing_curve(value[0][..., 2], value[1], label=key, ax=ax, windings=True)

ax.set_xlabel('Impact factor [M]')
ax.set_ylabel('Number of windings')
ax.legend()
plt.show()